# 04 — Tạo biến dẫn xuất và kết hợp các bảng dữ liệu

## Đơn vị quan sát và quy tắc nối bảng

Notebook đọc các bảng trung gian đã làm sạch, tạo biến dẫn xuất cho tiền điện tử và kết hợp dữ liệu ở tần suất phù hợp. Mỗi bảng giữ một đơn vị quan sát rõ ràng:

| Bảng | Một dòng tương ứng với | Mục đích |
|---|---|---|
| `fact_crypto_daily` | Một ticker–ngày có giá hợp lệ | Giá, chỉ số on-chain, trường thị trường và biến dẫn xuất |
| `fact_market_daily` | Một ticker–ngày có vốn hóa hoặc volume hợp lệ | Hai loại vốn hóa, volume spot và thị phần vốn hóa hiện tại |
| `fact_macro_daily` | Một ngày có quan sát vĩ mô | Các biến được quan sát ở tần suất ngày |
| `fact_macro_monthly` | Một tháng lịch | Chuỗi nguồn theo tháng và các biến ngày đã tổng hợp |
| `analysis_crypto_macro_monthly` | Một tháng trong lịch phân tích | Giá/lợi suất BTC, ETH, bối cảnh thị trường và vĩ mô |
| `fact_country_yearly` | Một ISO3–năm | Bối cảnh kinh tế đã tạo ở notebook 03, được giữ riêng |

Bối cảnh thị trường theo ngày được nối vào bảng ticker–ngày với quan hệ nhiều–một. Các bảng phân tích theo tháng được nối một–một trên `month`. Dữ liệu quốc gia theo năm không được ghép vào bảng crypto theo ngày; các biến vĩ mô tháng cũng không được điền tiếp xuống từng ngày.

Xếp hạng, thị phần và tỷ trọng vốn hóa BTC đều dùng `market_cap_current_usd`. Vốn hóa ước tính được giữ riêng.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd() / "scripts"))
from common import STAGING, TABLES, QUALITY, ensure_output_dirs, assert_unique_key, last_complete_month, write_csv_atomic
ensure_output_dirs()

crypto = pd.read_csv(STAGING / "crypto_base.csv.gz", parse_dates=["date"], low_memory=False)
market = pd.read_csv(STAGING / "market_clean.csv.gz", parse_dates=["date"], low_memory=False)
macro_daily = pd.read_csv(STAGING / "macro_daily.csv", parse_dates=["date"])
macro_monthly = pd.read_csv(STAGING / "macro_monthly.csv", parse_dates=["month"])
dim_coin = pd.read_csv(
    TABLES / "dim_coin.csv",
    parse_dates=["raw_first_date", "raw_last_date", "price_first_date", "price_last_date", "market_cap_as_of_date"],
)
for name, frame, key in [
    ("crypto_base", crypto, ["ticker", "date"]),
    ("market_clean", market, ["ticker", "date"]),
    ("macro_daily", macro_daily, ["date"]),
    ("macro_monthly", macro_monthly, ["month"]),
    ("dim_coin", dim_coin, ["ticker"]),
]:
    assert_unique_key(frame, key, name)

## Tính các biến dẫn xuất trong từng tài sản

Dữ liệu được sắp theo `ticker` và `date` trước khi tính toán. Lợi suất đơn là phần trăm thay đổi giá giữa hai quan sát kế tiếp; lợi suất log là chênh lệch log tự nhiên của hai mức giá.

Các đường trung bình dùng **7, 30 và 90 quan sát**. Biến động dùng độ lệch chuẩn mẫu của lợi suất log trong **30 quan sát**, nhân $\sqrt{365}\times 100$ để biểu diễn theo phần trăm năm hóa. Vì bảng chỉ giữ dòng có giá hợp lệ, cửa sổ theo số quan sát có thể dài hơn số ngày tương ứng khi dữ liệu bị khuyết; tên trường có chữ “daily” hoặc “30d” không tự bảo đảm lịch quan sát liên tục.

Drawdown là phần trăm chênh lệch giữa giá hiện tại và mức giá cao nhất đã xuất hiện trong lịch sử của chính tài sản. Tỷ lệ `turnover_ratio` bằng volume spot được báo cáo chia cho vốn hóa hiện tại cùng dòng; kết quả không hữu hạn được để thiếu.

Để gắn cờ lợi suất cực đoan, chương trình so lợi suất hiện tại với median và MAD của cửa sổ tối đa 365 quan sát trước đó, yêu cầu tối thiểu 60 quan sát không thiếu. Thao tác `shift(1)` loại quan sát đang xét khỏi mẫu tham chiếu. Điểm robust z dùng hệ số chuẩn hóa 1,4826; `|z| > 6` được gắn cờ nhưng không xóa hay điều chỉnh lợi suất. Khi chưa tính được điểm z, cờ `False` không đủ để kết luận quan sát không bất thường.

In [2]:
crypto_daily = crypto.sort_values(["ticker", "date"]).reset_index(drop=True)
grouped = crypto_daily.groupby("ticker", group_keys=False)
crypto_daily["log_return"] = grouped["price_usd"].transform(lambda s: np.log(s).diff())
crypto_daily["daily_return_pct"] = grouped["price_usd"].pct_change(fill_method=None) * 100
crypto_daily["ma7_usd"] = grouped["price_usd"].transform(lambda s: s.rolling(7, min_periods=7).mean())
crypto_daily["ma30_usd"] = grouped["price_usd"].transform(lambda s: s.rolling(30, min_periods=30).mean())
crypto_daily["ma90_usd"] = grouped["price_usd"].transform(lambda s: s.rolling(90, min_periods=90).mean())
crypto_daily["volatility_30d_annualized_pct"] = grouped["log_return"].transform(
    lambda s: s.rolling(30, min_periods=30).std() * np.sqrt(365) * 100
)
crypto_daily["drawdown_pct"] = grouped["price_usd"].transform(
    lambda s: (s / s.cummax() - 1) * 100
)
crypto_daily["turnover_ratio"] = (
    crypto_daily["reported_spot_volume_usd"] / crypto_daily["market_cap_current_usd"]
)
crypto_daily.loc[~np.isfinite(crypto_daily["turnover_ratio"]), "turnover_ratio"] = np.nan

def past_robust_z(series, window=365, min_periods=60):
    past = series.shift(1)
    rolling = past.rolling(window, min_periods=min_periods)
    median = rolling.median()
    mad = rolling.apply(
        lambda values: np.median(np.abs(values - np.median(values))), raw=True
    )
    return (series - median) / (1.4826 * mad.replace(0, np.nan))

crypto_daily["return_robust_z_365"] = grouped["daily_return_pct"].transform(past_robust_z)
crypto_daily["is_return_outlier"] = crypto_daily["return_robust_z_365"].abs().gt(6)
crypto_daily["is_up_day"] = crypto_daily["daily_return_pct"].gt(0).where(
    crypto_daily["daily_return_pct"].notna()
).astype("boolean")

assert crypto_daily["drawdown_pct"].dropna().le(1e-10).all()
assert crypto_daily["volatility_30d_annualized_pct"].dropna().ge(0).all()
display(crypto_daily.head())

,date,ticker,price_usd,market_cap_current_usd,market_cap_estimated_usd,reported_spot_volume_usd,active_addresses,tx_count,transfer_count,hashrate,...,daily_return_pct,ma7_usd,ma30_usd,ma90_usd,volatility_30d_annualized_pct,drawdown_pct,turnover_ratio,return_robust_z_365,is_return_outlier,is_up_day
0,2020-12-26,1INCH,1.619328,2.428992e+09,1.053593e+08,1.502920e+08,12305.0,27081.0,33665.0,NaN,...,NaN,NaN,NaN,NaN,NaN,0.000000,0.061874,NaN,False,<NA>
1,2020-12-27,1INCH,1.072350,1.608525e+09,7.995606e+07,1.329511e+08,6016.0,13526.0,17187.0,NaN,...,-33.778078,NaN,NaN,NaN,NaN,-33.778078,0.082654,NaN,False,False
2,2020-12-28,1INCH,1.104674,1.657011e+09,7.971789e+07,5.989677e+07,3929.0,7700.0,9924.0,NaN,...,3.014309,NaN,NaN,NaN,NaN,-31.781944,0.036147,NaN,False,True
3,2020-12-29,1INCH,0.893790,1.340686e+09,6.682589e+07,6.162150e+07,3330.0,7326.0,10132.0,NaN,...,-19.090136,NaN,NaN,NaN,NaN,-44.804863,0.045963,NaN,False,False
4,2020-12-30,1INCH,1.101307,1.651960e+09,8.243729e+07,8.259668e+07,2581.0,6347.0,8394.0,NaN,...,23.217547,NaN,NaN,NaN,NaN,-31.989907,0.049999,NaN,False,True


## Tổng hợp vốn hóa hiện tại theo ngày

Tổng vốn hóa của mỗi ngày chỉ cộng những tài sản có `market_cap_current_usd` hợp lệ. `assets_with_current_market_cap` ghi số tài sản đóng góp vào tổng đó.

Thị phần của một tài sản là vốn hóa hiện tại chia cho tổng cùng ngày, nhân 100. Tỷ trọng BTC dùng vốn hóa hiện tại của BTC làm tử số và cùng tổng vốn hóa làm mẫu số. Nếu thiếu vốn hóa BTC tại ngày đó, tỷ trọng BTC được để thiếu.

Tập tài sản có dữ liệu vốn hóa có thể thay đổi theo thời gian. Vì vậy các trường mang ý nghĩa `available_universe`: thị phần trong tập dữ liệu quan sát được, không đại diện cho toàn bộ thị trường tiền điện tử. Notebook kiểm tra tổng thị phần bằng 100% trong sai số cho phép và tỷ trọng BTC nằm trong khoảng 0–100%.

In [3]:
market_daily = market.sort_values(["ticker", "date"]).reset_index(drop=True)
current_rows = market_daily["market_cap_current_usd"].notna()
daily_summary = (
    market_daily.loc[current_rows]
    .groupby("date", as_index=False)
    .agg(
        total_current_market_cap_available_usd=("market_cap_current_usd", "sum"),
        assets_with_current_market_cap=("ticker", "nunique"),
    )
)
btc_current = (
    market_daily.loc[
        market_daily["ticker"].eq("BTC") & market_daily["market_cap_current_usd"].notna(),
        ["date", "market_cap_current_usd"],
    ]
    .rename(columns={"market_cap_current_usd": "btc_current_market_cap_usd"})
)
daily_summary = daily_summary.merge(btc_current, on="date", how="left", validate="one_to_one")
daily_summary["btc_dominance_current_available_universe_pct"] = (
    daily_summary["btc_current_market_cap_usd"]
    / daily_summary["total_current_market_cap_available_usd"]
    * 100
)

market_daily = market_daily.merge(daily_summary, on="date", how="left", validate="many_to_one")
market_daily["current_market_share_available_pct"] = (
    market_daily["market_cap_current_usd"]
    / market_daily["total_current_market_cap_available_usd"]
    * 100
)
share_sums = (
    market_daily.dropna(subset=["market_cap_current_usd"])
    .groupby("date")["current_market_share_available_pct"]
    .sum()
)
assert np.allclose(share_sums, 100, atol=1e-8)
assert daily_summary["btc_dominance_current_available_universe_pct"].dropna().between(0, 100).all()

crypto_daily = crypto_daily.merge(
    daily_summary[
        [
            "date", "btc_dominance_current_available_universe_pct",
            "total_current_market_cap_available_usd", "assets_with_current_market_cap",
        ]
    ],
    on="date", how="left", validate="many_to_one",
)
display(daily_summary.tail())

,date,total_current_market_cap_available_usd,assets_with_current_market_cap,btc_current_market_cap_usd,btc_dominance_current_available_universe_pct
5784,2026-05-19,2.287049e+12,32,1.538569e+12,67.273136
5785,2026-05-20,2.302990e+12,32,1.550639e+12,67.331583
5786,2026-05-21,2.309028e+12,32,1.554420e+12,67.319257
5787,2026-05-22,2.252610e+12,32,1.513835e+12,67.203586
5788,2026-05-23,2.283461e+12,32,1.534955e+12,67.220562


## Tạo bảng phân tích tiền điện tử–vĩ mô theo tháng

Với mỗi tài sản, chương trình chọn giá hợp lệ cuối cùng xuất hiện trong từng tháng. Lợi suất tháng được tính giữa các mức giá tháng kế tiếp có trong dữ liệu của tài sản. Nếu thiếu hẳn một tháng của tài sản, phép tính này có thể nối qua tháng khuyết vì được thực hiện trước khi ghép lịch tháng đầy đủ.

Bảng phân tích bắt đầu từ `2013-01` và kết thúc tại tháng hoàn chỉnh xác định từ ngày lớn nhất trong toàn bộ bảng giá crypto. Nếu ngày này chưa đến cuối tháng, tháng đang dở dang bị loại. Điều kiện đó giới hạn mốc thời gian, không bảo đảm mọi tài sản hoặc biến vĩ mô đều đủ dữ liệu trong từng tháng.

Trên lịch tháng này, BTC, ETH, bối cảnh thị trường ở ngày cuối cùng có dữ liệu trong tháng và bảng vĩ mô được nối trái với quan hệ một–một. Tháng không khớp hoặc chỉ tiêu chưa có dữ liệu vẫn được giữ với giá trị thiếu. Giá crypto là giá hợp lệ cuối cùng trong tháng, trong khi mỗi nguồn vĩ mô giữ quy ước quan sát riêng.

In [4]:
crypto_month_end = (
    crypto_daily.assign(month=crypto_daily["date"].dt.to_period("M").dt.to_timestamp("M"))
    .sort_values(["ticker", "date"])
    .groupby(["ticker", "month"], as_index=False)
    .tail(1)[["month", "ticker", "date", "price_usd"]]
    .rename(columns={"date": "crypto_observation_date"})
    .sort_values(["ticker", "month"])
)
crypto_month_end["monthly_return_pct"] = (
    crypto_month_end.groupby("ticker")["price_usd"].pct_change(fill_method=None) * 100
)

def asset_monthly(ticker, prefix):
    return (
        crypto_month_end.loc[
            crypto_month_end["ticker"].eq(ticker),
            ["month", "price_usd", "monthly_return_pct"],
        ]
        .rename(
            columns={
                "price_usd": f"{prefix}_price_usd",
                "monthly_return_pct": f"{prefix}_return_pct",
            }
        )
    )

btc_monthly = asset_monthly("BTC", "btc")
eth_monthly = asset_monthly("ETH", "eth")
market_monthly = (
    daily_summary.assign(month=daily_summary["date"].dt.to_period("M").dt.to_timestamp("M"))
    .sort_values("date")
    .groupby("month", as_index=False)
    .tail(1)[
        [
            "month", "btc_dominance_current_available_universe_pct",
            "total_current_market_cap_available_usd", "assets_with_current_market_cap",
        ]
    ]
)

last_complete = last_complete_month(crypto_daily["date"].max())
month_index = pd.DataFrame(
    {"month": pd.date_range("2013-01-31", last_complete, freq="ME")}
)
analysis = month_index.merge(btc_monthly, on="month", how="left", validate="one_to_one")
analysis = analysis.merge(eth_monthly, on="month", how="left", validate="one_to_one")
analysis = analysis.merge(market_monthly, on="month", how="left", validate="one_to_one")
analysis = analysis.merge(macro_monthly, on="month", how="left", validate="one_to_one")
assert_unique_key(analysis, ["month"], "analysis_crypto_macro_monthly")
assert analysis["month"].equals(month_index["month"])
assert analysis["month"].max() == last_complete
display(analysis.tail())

,month,btc_price_usd,btc_return_pct,eth_price_usd,eth_return_pct,btc_dominance_current_available_universe_pct,total_current_market_cap_available_usd,assets_with_current_market_cap,sp500_level,sp500_pe10,...,vix_max,vix_observations,brent_usd_per_barrel,dxy_reconstructed,vix_change_pct,risk_regime,brent_return_pct,dxy_return_pct,real_rate_pct,real_rate_change_pp
155,2025-12-31,87516.978038,-3.411105,2968.128014,-1.148409,65.521050,2.667355e+12,33,6853.03,NaN,...,17.62,22.0,61.35,98.361766,-21.352681,Neutral,-4.245357,-1.110472,1.462919,0.108004
156,2026-01-31,78702.385503,-10.071866,2444.243183,-17.650345,65.904351,2.386320e+12,33,6929.12,NaN,...,20.09,21.0,72.25,96.835725,4.057489,Neutral,17.766911,-1.551458,1.823569,0.360649
157,2026-02-28,66965.635309,-14.912826,1963.488610,-19.668852,64.968571,2.061063e+12,33,6893.81,NaN,...,21.77,20.0,71.32,97.621235,18.715270,Neutral,-1.287197,0.811178,1.715887,-0.107682
158,2026-03-31,68214.868039,1.865483,2101.431893,7.025418,64.977477,2.100651e+12,32,6654.42,NaN,...,31.05,22.0,126.69,100.195962,33.268174,Risk-off,77.636007,2.637466,0.993580,-0.722307
159,2026-04-30,76299.616129,11.851886,2256.615400,7.384656,66.621199,2.293156e+12,32,6957.01,NaN,...,25.78,21.0,124.24,98.163318,-22.600127,Neutral,-1.933854,-2.028669,0.509155,-0.484424


## Lưu bảng kết quả và kiểm tra phép nối

Notebook ghi năm bảng: `fact_crypto_daily.csv.gz`, `fact_market_daily.csv.gz`, `fact_macro_daily.csv`, `fact_macro_monthly.csv` và `analysis_crypto_macro_monthly.csv` vào `data/processed/tables/`.

Báo cáo `join_audit.csv` tóm tắt ba phép nối: bảng giá với bối cảnh thị trường theo ngày, lịch tháng với BTC và bảng phân tích tháng với vĩ mô. Mỗi dòng ghi đơn vị quan sát, quan hệ khóa, số dòng trước/sau và số dòng bên trái không khớp khóa.

Tham số `validate` trong các phép nối kiểm tra quan hệ một–một hoặc nhiều–một theo thiết kế. Khóa không khớp vẫn có thể tạo giá trị thiếu trong phép nối trái; điều này được ghi nhận riêng, không bị coi là lỗi trùng khóa.

In [5]:
fact_macro_daily = macro_daily.copy()
fact_macro_monthly = macro_monthly.copy()

write_csv_atomic(crypto_daily, TABLES / "fact_crypto_daily.csv.gz")
write_csv_atomic(market_daily, TABLES / "fact_market_daily.csv.gz")
write_csv_atomic(fact_macro_daily, TABLES / "fact_macro_daily.csv")
write_csv_atomic(fact_macro_monthly, TABLES / "fact_macro_monthly.csv")
write_csv_atomic(analysis, TABLES / "analysis_crypto_macro_monthly.csv")

crypto_context_check = crypto.merge(daily_summary[["date"]], on="date", how="left", validate="many_to_one", indicator=True)
macro_btc_check = month_index.merge(btc_monthly, on="month", how="left", validate="one_to_one", indicator=True)
join_audit = pd.DataFrame(
    [
        {
            "join_name": "crypto_price_to_daily_market_context",
            "left_grain": "ticker_date", "right_grain": "date", "cardinality": "many_to_one",
            "left_rows_before": len(crypto), "rows_after": len(crypto_context_check),
            "left_unmatched": int(crypto_context_check["_merge"].eq("left_only").sum()),
            "purpose": "attach current-cap available-universe context without row multiplication",
        },
        {
            "join_name": "calendar_month_to_btc_monthly",
            "left_grain": "month", "right_grain": "month", "cardinality": "one_to_one",
            "left_rows_before": len(month_index), "rows_after": len(macro_btc_check),
            "left_unmatched": int(macro_btc_check["_merge"].eq("left_only").sum()),
            "purpose": "align BTC last-valid-price monthly returns to an explicit complete calendar",
        },
        {
            "join_name": "analysis_month_to_macro_month",
            "left_grain": "month", "right_grain": "month", "cardinality": "one_to_one",
            "left_rows_before": len(month_index), "rows_after": len(analysis),
            "left_unmatched": int((~month_index["month"].isin(macro_monthly["month"])).sum()),
            "purpose": "combine transformed crypto and macro measures at the same monthly grain",
        },
    ]
)
write_csv_atomic(join_audit, QUALITY / "join_audit.csv")
display(join_audit)

,join_name,left_grain,right_grain,cardinality,left_rows_before,rows_after,left_unmatched,purpose
0,crypto_price_to_daily_market_context,ticker_date,date,many_to_one,123358,123358,0,attach current-cap available-universe context ...
1,calendar_month_to_btc_monthly,month,month,one_to_one,160,160,0,align BTC last-valid-price monthly returns to ...
2,analysis_month_to_macro_month,month,month,one_to_one,160,160,0,combine transformed crypto and macro measures ...
